### Criando esquema de estrela (tabelas Fatos e Dimensões)

In [0]:
from pyspark.sql import functions as F, Window

CATALOG = "workspace"
SCHEMA = "sinan"

df_detalhe = spark.table(f"{CATALOG}.{SCHEMA}.gold_sinan_ntep_detalhe")

Dimensão 1: características do notificante ---


In [0]:
dim_notificante = (df_detalhe
    .select("SEXO", "RACA")
    .distinct()
    .withColumn("SK_NOTIFICANTE", F.row_number().over(Window.orderBy("SEXO", "RACA")))
    .select("SK_NOTIFICANTE", "SEXO", "RACA")
)

Dimensão 2: características do local de notificação


In [0]:
dim_local = (df_detalhe
    .select("REGIAO_NOTIF", "UF_NOTIF", "COD_MUN_NOTIF")
    .distinct()
    .withColumn("SK_LOCAL", F.row_number().over(
        Window.orderBy("REGIAO_NOTIF", "UF_NOTIF", "COD_MUN_NOTIF")))
    .select("SK_LOCAL", "REGIAO_NOTIF", "UF_NOTIF", "COD_MUN_NOTIF")
)

Dimensão 3: características da notificação em si ---


In [0]:
dim_notificacao = (df_detalhe
    .select("ANO_NOTIF", "CID10", "CAPITULO_CID_OFICIAL", "AGRAVO", "CATEGORIA_NTEP", "PRESENTE_NO_NTEP")
    .distinct()
    .withColumn("SK_NOTIFICACAO", F.row_number().over(
        Window.orderBy("ANO_NOTIF", "AGRAVO", "CID10")))
    .select("SK_NOTIFICACAO", "ANO_NOTIF", "CID10", "CAPITULO_CID_OFICIAL", "AGRAVO",
            "CATEGORIA_NTEP", "PRESENTE_NO_NTEP")
)

Fato: junta o detalhe às três dimensões, agrega e recebe PK sintética 

In [0]:
fato = (df_detalhe
    .join(F.broadcast(dim_notificante), on=["SEXO", "RACA"], how="left")
    .join(F.broadcast(dim_local), on=["REGIAO_NOTIF", "UF_NOTIF", "COD_MUN_NOTIF"], how="left")
    .join(F.broadcast(dim_notificacao),
          on=["ANO_NOTIF", "CID10", "CAPITULO_CID_OFICIAL", "AGRAVO", "CATEGORIA_NTEP", "PRESENTE_NO_NTEP"],
          how="left")
    .groupBy("SK_NOTIFICANTE", "SK_LOCAL", "SK_NOTIFICACAO")
    .agg(F.count(F.lit(1)).alias("N_NOTIFICACOES"))
    .withColumn("SK_FATO", F.row_number().over(
        Window.orderBy("SK_NOTIFICANTE", "SK_LOCAL", "SK_NOTIFICACAO")))
    .select("SK_FATO", "SK_NOTIFICANTE", "SK_LOCAL", "SK_NOTIFICACAO", "N_NOTIFICACOES")
)

Grava as 4 tabelas ---


In [0]:
dim_notificante.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_notificante")
dim_local.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_local_notificacao")
dim_notificacao.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_notificacao")
fato.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.fato_notificacoes_ntep")

for tabela in ["dim_notificante", "dim_local_notificacao", "dim_notificacao", "fato_notificacoes_ntep"]:
    print(f"{tabela}: {spark.table(f'{CATALOG}.{SCHEMA}.{tabela}').count():,} linhas")